In [2]:
# Setup
%load_ext autoreload
%autoreload 2
#%load_ext jupyter_ai_magics
import sys; sys.path.append('..')
from src.env_setup import init_analysis_env; init_analysis_env()

df_ = load_local_data("Stock Screener Data*.csv*", na_values=['-'], dtype={'Code': str})
df_.head()

# Cleaning

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
🚀 Analysis environment initialized: pd, np, yf, plt, sns, load_local_data, ai, datetime, timedelta, xw are ready.
Loading: /home/mrsmmori/notebooks/Win_Downloads/Stock Screener Data 7 Jun 11-8-22.csv.csv


,Trading Name,Code,Last Price,ROE %,Mkt Cap ($M),Tot. Rev ($M),P/E,Yield (%),Sector,GTI Score,4-wk %Pr. Chg.,13-wk %Pr. Chg.,26-wk %Pr. Chg.,52-wk %Pr. Chg.,Net Profit,Price/CF,Debt/Equity,1-yr %Rev. Chg.,Price/Book Value
0,Tai Sin Electric,500,0.55,12.09,SGD 250.843,SGD 527.837,14.30,4.31,Industrials,78.00,-5.22,5.24,-5.56,38.66,0.05,NaN,0.04,19.98,1.12
1,HS Optimus,504,0.01,5.73,SGD 37.664,SGD 13.551,13.73,NaN,Consumer Cyclical,67.00,0.00,133.33,250.00,250.00,0.20,74.07,0.01,-6.18,0.77
2,AsiaMedic,505,0.02,13.34,SGD 26.4,SGD 35.221,13.14,NaN,Healthcare,66.00,0.00,21.05,21.05,91.67,0.03,6.85,0.96,21.81,1.64
3,Fuji Offset,508,0.73,2.33,SGD 43.736,SGD 3.274,48.34,0.68,Industrials,70.00,28.95,5.76,36.11,-10.91,0.28,357.14,0.00,-9.76,1.14
4,DISA,532,0.00,-137.72,SGD 14.09,SGD 2.667,NaN,NaN,Technology,34.00,0.00,0.00,0.00,0.00,-0.42,NaN,0.01,-19.20,3.67


In [3]:
df = df_.copy()

In [4]:
rename_dict = {
    'Trading Name':    'name',
    'Code':            'code',
    'Last Price':      'price',
    'ROE %':           'roe',
    'Mkt Cap ($M)':    'mkt_cap',
    'Tot. Rev ($M)':   'rev',
    'P/E':             'pe',
    'Yield (%)':       'yield',
    'Sector':          'sector',
    'GTI Score':       'gti',
    '4-wk %Pr. Chg.':  'chg_4w',
    '13-wk %Pr. Chg.': 'chg_13w',
    '26-wk %Pr. Chg.': 'chg_26w',
    '52-wk %Pr. Chg.': 'chg_52w',
    'Net Profit':      'net_profit',
    'Price/CF':        'p_cf',
    'Debt/Equity':     'de',
    '1-yr %Rev. Chg.': 'rev_chg_1y',
    'Price/Book Value':'pb'
}

df = df.rename(columns=rename_dict)

In [5]:
df['mkt_cap'] = pd.to_numeric(df['mkt_cap'], errors='coerce')
df['mkt_cap'] = df['mkt_cap'].fillna(0)
cap_cutoff = df['mkt_cap'].quantile(0.30)  # 下位30%の足切りライン
df = df[df['mkt_cap'] >= cap_cutoff]
df

,name,code,price,roe,mkt_cap,rev,pe,yield,sector,gti,chg_4w,chg_13w,chg_26w,chg_52w,net_profit,p_cf,de,rev_chg_1y,pb
0,Tai Sin Electric,500,0.55,12.09,0.00,SGD 527.837,14.30,4.31,Industrials,78.00,-5.22,5.24,-5.56,38.66,0.05,NaN,0.04,19.98,1.12
1,HS Optimus,504,0.01,5.73,0.00,SGD 13.551,13.73,NaN,Consumer Cyclical,67.00,0.00,133.33,250.00,250.00,0.20,74.07,0.01,-6.18,0.77
2,AsiaMedic,505,0.02,13.34,0.00,SGD 35.221,13.14,NaN,Healthcare,66.00,0.00,21.05,21.05,91.67,0.03,6.85,0.96,21.81,1.64
3,Fuji Offset,508,0.73,2.33,0.00,SGD 3.274,48.34,0.68,Industrials,70.00,28.95,5.76,36.11,-10.91,0.28,357.14,0.00,-9.76,1.14
4,DISA,532,0.00,-137.72,0.00,SGD 2.667,NaN,NaN,Technology,34.00,0.00,0.00,0.00,0.00,-0.42,NaN,0.01,-19.20,3.67
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
610,Zheneng Jinjiang,BWM,0.58,9.24,0.00,"SGD 3,784.87",6.13,6.32,Utilities,68.90,6.32,5.42,35.22,36.70,0.20,2.91,0.95,1.44,0.55
611,Zhongmin Baihui,5SR,0.42,20.54,0.00,SGD 960.396,14.77,2.38,Consumer Cyclical,74.90,-6.67,-17.65,-9.68,-24.56,0.03,3.44,0.97,-3.30,1.87
612,Zhongxin Fruit,5EG,0.03,19.54,0.00,SGD 165.24,8.58,NaN,Consumer Defensive,68.50,-5.71,-5.71,-13.16,3.13,0.12,26.60,NaN,76.44,1.02
613,ZICO Hldgs,40W,0.05,3.83,0.00,SGD 14.276,NaN,2.04,Industrials,76.30,16.67,-28.57,8.70,19.05,-0.47,NaN,0.03,2.06,0.72


In [5]:
df['rev'].sort_values()

608     CNY 28,504.82
517       EUR 214.617
278        EUR 50.434
173        GBP 38.305
532    HKD 12,044.085
            ...      
351               NaN
450               NaN
455               NaN
456               NaN
560               NaN
Name: rev, Length: 615, dtype: str

In [6]:
fx_rates = {
    'SGD': 1.0,
    'GBP': 1.70,
    'USD': 1.35,  # 1 USD = 1.35 SGD
    'EUR': 1.45,  # 1 EUR = 1.45 SGD
    'CNY': 0.19,  # 1 CNY = 0.19 SGD
    'HKD': 0.17,  # 1 HKD = 0.17 SGD
    'MYR': 0.29,  # 1 MYR = 0.29 SGD
}

def convert_rev_to_sgd(row_value):
    val_str = str(row_value).strip().upper()
    if val_str in ['NAN', 'NONE', '-', '']:
        return 0.0
    
    detected_currency = 'SGD'
    

    for curr in fx_rates.keys():
        if curr in val_str:
            detected_currency = curr
            val_str = val_str.replace(curr, '') 
            break
            

    val_str = val_str.replace(',', '').strip()
    try:
        num_val = float(val_str)
    except ValueError:
        return 0.0
        

    return num_val * fx_rates[detected_currency]

df['rev_sgd'] = df['rev'].apply(convert_rev_to_sgd)

df_top5 = df.sort_values(by='rev_sgd', ascending=False).groupby('sector').head(5)

df_top5 = df_top5.sort_values(by=['sector', 'rev_sgd'], ascending=[True, False])

result_cols = ['sector', 'name', 'code', 'rev', 'rev_sgd', 'mkt_cap']
df_top5[result_cols]

,sector,name,code,rev,rev_sgd,mkt_cap
322,Basic Materials,Le Tree Holdings,E6R,"SGD 283,025",283025.00,0.00
509,Basic Materials,Sri Trang Agro,NC2,"SGD 105,934.507",105934.51,0.00
510,Basic Materials,Sri Trang Gloves,STG,"SGD 22,829.589",22829.59,0.00
133,Basic Materials,ChinaSunsine,QES,"SGD 3,277.392",3277.39,0.00
230,Basic Materials,Halcyon Agri,5VJ,"SGD 2,961.538",2961.54,0.00
...,...,...,...,...,...,...
483,Utilities,SIIC Environment,BHK,"SGD 7,072.781",7072.78,0.00
468,Utilities,Sembcorp Ind,U96,"SGD 5,799",5799.00,0.00
126,Utilities,China Everbright,U9E,"SGD 5,355.11",5355.11,0.00
610,Utilities,Zheneng Jinjiang,BWM,"SGD 3,784.87",3784.87,0.00
